# Stage 2 - Cleaning GPS points

**All point data in this notebook is SYNTHETIC** (`synthetic_test_fixture`, seed 42): positions are linearly
interpolated between real Delhi stop coordinates, timestamps are invented, and defects are injected on purpose
so the cleaning rules can be scored against known ground truth. No mobility conclusion may be drawn from it.
Real analysis starts when the official Delhi timetable or the Porto trajectories are in place.

Rules applied, in order: missing values -> duplicate pings -> city bounding box -> out-of-order timestamps ->
impossible speed (spike rule). Thresholds are constants in `src/cleaning.py`, each with its reasoning.

In [1]:
import json, pathlib, sys
sys.path.append(str(pathlib.Path.cwd().parent))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

from src.io import PROJECT_ROOT, load_gtfs
from src.synthetic import make_fixture_points
from src import cleaning as C

FEED_DIR = PROJECT_ROOT / "data" / "raw" / "delhi_gtfs_trimmed"
FIG = PROJECT_ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140)

In [2]:
# 1) Build the synthetic fixture (deterministic: random_state=42)
raw = make_fixture_points(FEED_DIR, random_state=42)
print(f"rows={len(raw):,}  vehicles={raw.vehicle_id.nunique():,}  trips={raw.trip_id.nunique():,}")
print("injected defects:", {c: int(raw[c].sum()) for c in raw.columns if c.startswith('defect_')})
raw.head(3)

rows=431,820  vehicles=2,403  trips=2,403
injected defects: {'defect_jump': 2156, 'defect_duplicate': 2148, 'defect_outside': 863, 'defect_time': 1295}


,vehicle_id,trip_id,route_id,timestamp,lat,lon,is_dwell,defect_jump,defect_duplicate,defect_outside,defect_time,data_source
0,V5371,5371_11_40,5371,2024-06-04 02:15:00,28.729024,77.044383,False,False,False,False,False,synthetic_test_fixture
1,V5811,5811_12_42,5811,2024-06-03 10:31:00,28.614908,77.084099,False,False,False,False,False,synthetic_test_fixture
2,V5519,5519_21_35,5519,2024-06-03 21:42:30,28.728144,77.160888,False,False,False,False,False,synthetic_test_fixture


In [3]:
# 2) Bounding box from the feed's own stops (+5.5 km margin), not a hard-coded rectangle
feed = load_gtfs(FEED_DIR)
bbox = C.bbox_from_stops(feed["stops"])
print({k: round(v, 5) for k, v in bbox.items() if isinstance(v, float)})

{'lat_min': 28.36665, 'lat_max': 28.92187, 'lon_min': 76.79277, 'lon_max': 77.46455, 'margin_deg': 0.05}


In [4]:
# 3) Clean
clean, report = C.clean_points(raw, bbox)
print(f"rows {report['rows_in']:,} -> {report['rows_out']:,}  ({report['pct_removed']}% removed)")
pd.Series({k: v for k, v in report.items() if isinstance(v, int)}).to_frame("rows")

rows 431,820 -> 426,845  (1.152% removed)


,rows
rows_in,431820
dropped_missing,0
dropped_exact_duplicate_pings,2123
dropped_conflicting_timestamps,36
dropped_duplicates_total,2159
dropped_outside_bbox,859
dropped_nonpositive_timestep,0
dropped_impossible_jumps,1957
jump_passes_used,1
residual_implausible_steps,69


In [5]:
# 4) How much of the injected damage did the rules actually catch?
scores = C.score_against_known_defects(raw, clean)
scores

{'defect_jump': {'injected': 2156, 'survived_cleaning': 210, 'recall': 0.9026},
 'defect_duplicate': {'injected': 2148,
  'duplicate_keys_remaining_in_clean': 0,
  'recall': 1.0},
 'defect_outside': {'injected': 863, 'survived_cleaning': 0, 'recall': 1.0},
 'defect_time': {'injected': 1295,
  'nonpositive_steps_after_clean': 0,
  'note': 'repaired by sorting; rows intentionally kept'}}

In [6]:
# 5) Speed before vs after. Log-x, because raw speeds span 0 to thousands of km/h.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, frame, title in [(axes[0], C.add_step_metrics(raw), "raw"),
                         (axes[1], clean, "cleaned")]:
    s = frame["speed_kmh"].dropna()
    s = s[s > 0]
    ax.hist(s, bins=np.logspace(-1, 4, 60), color="#111111", alpha=0.85)
    ax.set_xscale("log"); ax.set_xlabel("implied speed between pings (km/h)"); ax.set_title(title)
    ax.axvline(C.MAX_SPEED_KMH, color="#9A9A9A", ls="--", lw=1)
axes[0].set_ylabel("pings")
fig.suptitle("SYNTHETIC fixture - speed distribution, dashed line = 100 km/h threshold", y=1.03)
fig.tight_layout(); fig.savefig(FIG / "stage2_speed_before_after.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage2_speed_before_after.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage2_speed_before_after.png


In [7]:
# 6) Where were the removed points? (map check: they should be strays, not a district)
key = ["vehicle_id", "timestamp", "lat", "lon"]
clean_keys = set(map(tuple, clean[key].itertuples(index=False, name=None)))
is_removed = ~np.fromiter((k in clean_keys for k in raw[key].itertuples(index=False, name=None)),
                          bool, len(raw))

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(clean["lon"], clean["lat"], s=1, c="#D9D9D9", linewidths=0, label=f"kept ({len(clean):,})")
ax.scatter(raw.loc[is_removed, "lon"], raw.loc[is_removed, "lat"], s=6, c="#111111",
           linewidths=0, label=f"removed ({int(is_removed.sum()):,})")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("SYNTHETIC fixture: kept vs removed points")
ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(FIG / "stage2_removed_points.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage2_removed_points.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage2_removed_points.png


In [8]:
# 7) Persist outputs. 'synthetic' is recorded in the report and in the file names.
raw_path = PROJECT_ROOT / "data" / "interim" / "points_synthetic_raw.parquet"
clean_path = PROJECT_ROOT / "data" / "processed" / "points_synthetic_cleaned.parquet"
raw.to_parquet(raw_path, index=False)
clean.to_parquet(clean_path, index=False)

out = {
    "stage": 2,
    "data_source": "synthetic_test_fixture",
    "synthetic": True,
    "seed": 42,
    "feed_dir": str(FEED_DIR),
    "report": report,
    "known_defect_scoring": scores,
    "outputs": {"raw": str(raw_path), "cleaned": str(clean_path)},
}
p = PROJECT_ROOT / "outputs" / "cleaning_report.json"
p.write_text(json.dumps(out, indent=2, default=str))
print("wrote", p)

wrote /home/ashira/Documents/Projects/Fleetsense/outputs/cleaning_report.json


## Reading the result

* `recall` on `defect_jump` is below 1 because an injected offset drawn from a Gaussian can be only a few metres,
  which is indistinguishable from GPS noise. Those points are **correctly kept** - deleting them would delete real
  dwell pings.
* `defect_time` is not scored by recall: sorting the table repairs order and the observation itself is valid.
  What matters is that no non-positive time step survives (`nonpositive_steps_after_clean = 0`).
* `residual_implausible_steps` is the honest admission: when a step is impossible, position alone cannot say which
  of the two pings is wrong, so the step is flagged (`step_implausible`) and Stage 3 excludes it from speed features.